# 02 ― ECMWF GRIB2 を直接開く

`aiseed-weather` が S3 (`s3://ecmwf-forecasts`) からキャッシュした GRIB2 ファイルを、自前のコードで読んでみる。

アプリの天気図タブで一度でも「更新」を押せば、以下のような階層でキャッシュが残る:

```
<data_dir>/ecmwf/{YYYYMMDD}/{HH}z/{param}_{step}h.grib2
```

(例: `<data_dir>/ecmwf/20260518/00z/msl_0h.grib2` ― 2026-05-18 00z 解析の MSL)

**データ帰属**: 出典 ECMWF Open Data, CC-BY-4.0


In [ ]:
from pathlib import Path
from aiseed_weather.models.user_settings import load_or_init, resolved_data_dir

data_dir = resolved_data_dir(load_or_init().settings)
ecmwf_root = data_dir / "ecmwf"
print(f"ECMWF cache root: {ecmwf_root}")


## 最新のキャッシュ済みランを探す


In [ ]:
# 日付ディレクトリ (YYYYMMDD) を新しい順に並べ、最初に見つかった run を選ぶ
runs: list[Path] = []
if ecmwf_root.exists():
    for date_dir in sorted(ecmwf_root.iterdir(), reverse=True):
        if not date_dir.is_dir():
            continue
        for hh_dir in sorted(date_dir.iterdir(), reverse=True):
            if hh_dir.is_dir() and any(hh_dir.glob("*.grib2")):
                runs.append(hh_dir)

if not runs:
    print("(キャッシュにまだ GRIB2 がありません。アプリで天気図を一度表示してください)")
else:
    latest = runs[0]
    print(f"最新ラン: {latest.relative_to(data_dir)}")
    for f in sorted(latest.glob("*.grib2"))[:10]:
        print(f"  - {f.name}  ({f.stat().st_size // 1024} KB)")


## MSL (海面更正気圧) を xarray で開く

`cfgrib` エンジンに渡すだけで xarray Dataset が手に入る。


In [ ]:
import xarray as xr

msl_path = next(latest.glob("msl_*.grib2"), None)
if msl_path is None:
    print("MSL のキャッシュがこのランにはありません。別のラン or 別のパラメータを試してください。")
else:
    ds = xr.open_dataset(msl_path, engine="cfgrib")
    print(ds)


## cartopy で日本周辺をプロット


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

msl = ds["msl"] / 100.0  # Pa → hPa

fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.LambertConformal(
    central_longitude=137.5, central_latitude=37.5,
))
ax.set_extent([120, 155, 22, 48], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="#444")

# 等値線 (4 hPa 間隔)
levels = range(960, 1056, 4)
cs = ax.contour(
    msl.longitude, msl.latitude, msl.values,
    levels=levels, transform=ccrs.PlateCarree(),
    colors="#234b86", linewidths=0.8,
)
ax.clabel(cs, inline=True, fontsize=8, fmt="%d")

ax.set_title(f"MSL  {ds.time.values}  (出典: ECMWF Open Data, CC-BY-4.0)")
plt.show()


## 自由解析のヒント

- 同じキャッシュには `t2m_*.grib2` (2 m 気温), `tp_*.grib2` (積算降水量),
  `u10_*.grib2 / v10_*.grib2` (10 m 風) などもある
- `xr.open_mfdataset(latest.glob("msl_*.grib2"))` で複数 step を一括で開ける
- 差分 (例: 24h 気圧変化) も `ds.diff('step')` で簡単に取れる
- GRIB2 のメタデータは `ds.attrs` / 各変数の `attrs` に入っている

**注意**: 出力した図を公開する場合は、必ず帰属 (出典 ECMWF, CC-BY-4.0)
を画像内に入れること。`figures/footer.py` を参考に。
